# Chapter 9: Generative Recommendation & Fine-Tuning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kimfalk/modern-recommender-systems/blob/main/notebooks/chapter-09/gen_recsys.ipynb)

This notebook accompanies the chapter on generative recommendation. It covers:

1. **Preparing the data** — converting semantic IDs into prefixed tokens and user histories into token sequences (Listings 9.4–9.6)
2. **Fine-tuning DistilGPT2** on those sequences with a custom dataset (Listings 9.7–9.9)
3. **Prompt enrichment** — request context, user attributes, and explicit feedback tokens (Listings 9.10–9.14)
4. **Generating recommendations** and decoding semantic IDs back to titles (Listings 9.16–9.18)
5. **Evaluation** — generative hit rate and perplexity ranking (Listings 9.19–9.20)

Advanced training strategies (contrastive loss, DPO, ICL reranking, anchored prompts, sampled softmax) live in the companion notebook `gen_recsys_alignment.ipynb`.

> **Prerequisite:** this notebook consumes the semantic IDs built in the Semantic IDs chapter (`item_df` with a `final_id` tuple per item). If that artifact is not available, a synthetic fallback is generated below so every cell still runs.


In [ ]:
# Environment Setup
from recsys.utils.colab import setup_colab_environment, get_data_path, check_gpu

# One-line setup for Colab users
setup_colab_environment()

# Check GPU availability
check_gpu()

In [ ]:
# Imports
import random
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
from transformers import (
    GPT2LMHeadModel,
    GPT2Tokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

DATA_PATH = get_data_path()
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## Loading the data

We need two things from earlier chapters:

- `item_df` — one row per catalog item with columns `id` (UUID), `title`, and `final_id` (the semantic ID tuple, e.g. `(8, 12, 145, 0)`)
- `training_data` — one row per user with a `history` column (list of item UUIDs, in interaction order)

The first cell tries to load the real artifacts produced by the Semantic IDs chapter. If they are missing, the second cell builds a small synthetic catalog so the rest of the notebook is runnable end to end.

In [ ]:
# Try to load the semantic ID artifacts from the Semantic IDs chapter
import os

ITEM_PATH = os.path.join(DATA_PATH, "semantic_ids", "items_with_semantic_ids.parquet")
HIST_PATH = os.path.join(DATA_PATH, "semantic_ids", "user_histories.parquet")

item_df = None
training_data = None

if os.path.exists(ITEM_PATH) and os.path.exists(HIST_PATH):
    item_df = pd.read_parquet(ITEM_PATH)
    # final_id is stored as a list in parquet; convert back to tuples for dict keys
    item_df["final_id"] = item_df["final_id"].apply(tuple)
    training_data = pd.read_parquet(HIST_PATH)
    training_data["history"] = training_data["history"].apply(list)
    print(f"Loaded {len(item_df)} items and {len(training_data)} user histories.")
else:
    print("Semantic ID artifacts not found - falling back to synthetic data.")

In [ ]:
# Synthetic fallback: a small catalog with hierarchical semantic IDs
# (codebook sizes [16, 32, 128], matching the Semantic IDs chapter)
if item_df is None:
    n_items = 500
    n_users = 400

    rows = []
    seen = set()
    for i in range(n_items):
        # Correlated levels so the hierarchy is learnable
        l1 = np.random.randint(0, 16)
        l2 = (l1 * 2 + np.random.randint(0, 2)) % 32
        l3 = (l2 * 4 + np.random.randint(0, 4)) % 128
        leaf = 0
        while (l1, l2, l3, leaf) in seen:
            leaf += 1
        seen.add((l1, l2, l3, leaf))
        rows.append({
            "id": f"uuid_{i:03d}",
            "title": f"Movie {i:03d}",
            "final_id": (l1, l2, l3, leaf),
            "genre_tags": f"CLUSTER_{l1}",
        })
    item_df = pd.DataFrame(rows)

    # Users watch mostly within one or two L1 clusters
    by_l1 = {}
    for _, r in item_df.iterrows():
        by_l1.setdefault(r["final_id"][0], []).append(r["id"])

    users = []
    for u in range(n_users):
        clusters = np.random.choice(list(by_l1.keys()), size=2, replace=False)
        pool = by_l1[clusters[0]] * 3 + by_l1[clusters[1]]
        length = np.random.randint(6, 20)
        history = list(np.random.choice(pool, size=length, replace=False))
        users.append({"uid": u, "history": history})
    training_data = pd.DataFrame(users)

print(item_df.head(3))
print(training_data.head(3))

In [ ]:
# Hold out the last 20% of users for evaluation
split = int(len(training_data) * 0.8)
train_data = training_data.iloc[:split].reset_index(drop=True)
test_data = training_data.iloc[split:].reset_index(drop=True)
print(f"{len(train_data)} training users, {len(test_data)} test users")

## 9.2 Preparing the Data: Sequences & Context

Each item's semantic ID tuple becomes four prefixed tokens — `L1_`, `L2_`, `L3_` for the codebook levels and `LF_` for the leaf that uniquely identifies the item. The prefixes stop the model from confusing `5` at level 1 with `5` at level 3, and `LF_` doubles as an end-of-item delimiter.

The `BaseFormatter` below combines Listings 9.4 (per-item token conversion) and 9.5 (full-history formatting).

In [ ]:
# Listing 9.4 + 9.5: Converting user histories to semantic ID sequences
class BaseFormatter:
    def __init__(self, item_df):
        self.item_map = dict(zip(item_df['id'], item_df['final_id']))  #A
        self.tokens_missing = Counter()

    def get_item_tokens(self, item_uuid):  #B
        if item_uuid not in self.item_map:
            return []
        s = self.item_map[item_uuid]
        return [
            f"L1_{s[0]}",
            f"L2_{s[1]}",
            f"L3_{s[2]}",
            f"LF_{s[3]}",
        ]

    def format(self, user_record, is_training=True):  #C
        tokens = []
        for t in user_record['history']:  #D
            if t in self.item_map:
                tokens.extend(self.get_item_tokens(t))
            else:
                self.tokens_missing.update([t])  #E
        return " ".join(tokens)

#A UUID -> semantic ID tuple lookup
#B Convert one item UUID to four tokens
#C Format an entire user history
#D Iterate over interaction history
#E Track unmapped items for debugging

formatter = BaseFormatter(item_df)
print(formatter.format(train_data.iloc[0])[:120], "...")

In [ ]:
# Listing 9.6: Collecting new tokens for the model vocabulary
def find_new_tokens(formatter, train_data):
    new_tokens = set()  #A
    for user_inx in range(train_data.shape[0]):  #B
        prompt = formatter.format(train_data.iloc[user_inx], is_training=False)
        new_tokens.update(prompt.split())  #C
    return new_tokens

#A Initialize empty token set
#B Scan all users
#C Collect unique tokens

new_tokens = find_new_tokens(formatter, train_data)
print(f"{len(new_tokens)} new semantic ID tokens")

### 9.2.1 Using a Pretrained Model

We fine-tune **DistilGPT2** — small enough to train on a laptop. The code works with any HuggingFace-compatible causal LM; only the model name needs to change. With a larger base model (Llama 3, Mistral), swap full fine-tuning for LoRA via the HuggingFace PEFT library.

In [ ]:
# Listing 9.7: Load model and extend vocabulary
model_name = "distilgpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)  #A
model = GPT2LMHeadModel.from_pretrained(model_name)  #B

tokenizer.add_tokens(list(new_tokens))  #C
model.resize_token_embeddings(len(tokenizer))  #D

#A Load pretrained tokenizer
#B Load pretrained model
#C Add semantic ID tokens to tokenizer
#D Resize embedding layer for new vocabulary

print(f"Vocabulary size after extension: {len(tokenizer)}")

In [ ]:
# Write each user's formatted sequence to a text file, one per line,
# with <|endoftext|> as a separator between users.
with open("train_recsys.txt", "w") as f:
    for i in range(len(train_data)):
        line = formatter.format(train_data.iloc[i])
        if line:
            f.write(line + " <|endoftext|>\n")

with open("train_recsys.txt") as f:
    print("Sample line:", f.readline()[:120], "...")

In [ ]:
# Listing 9.8: Configure the training pipeline
class SemanticIDDataset(Dataset):  #A
    def __init__(self, tokenizer, file_path, block_size=128):
        with open(file_path, 'r') as f:
            text = f.read()
        tokenized = tokenizer(
            text, truncation=True, max_length=block_size,
            return_overflowing_tokens=True, return_length=True,
        )
        self.examples = [
            torch.tensor(ids)
            for ids, length in zip(tokenized['input_ids'],
                                   tokenized['length'])
            if length == block_size  #B
        ]

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return {"input_ids": self.examples[idx],
                "labels": self.examples[idx]}  #C


train_dataset = SemanticIDDataset(
    tokenizer=tokenizer,
    file_path="train_recsys.txt",  #D
    block_size=128  #E
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=False  #F
)

training_args = TrainingArguments(
    output_dir="./recsys_gpt",
    overwrite_output_dir=True,
    num_train_epochs=2,  #G
    per_device_train_batch_size=2,
    learning_rate=5e-4,  #H
    save_steps=1000  #I
)

#A Custom dataset replacing deprecated TextDataset
#B Keep only full-length blocks
#C Labels equal input_ids for causal LM
#D Formatted user histories, one per line
#E Maximum sequence length in tokens
#F Causal LM mode (predict next token)
#G Number of passes over the dataset
#H Higher LR for new vocabulary
#I Checkpoint interval

print(f"{len(train_dataset)} training blocks of 128 tokens")

In [ ]:
# Listing 9.9: Fine-tune the model
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
)  #A

trainer.train()  #B
print("Fine-tuning complete. The model now speaks 'RecSys'!")

#A Instantiate trainer with all components
#B Start fine-tuning

### 9.2.2 Adding Request Context

A special token describing the request context — time of day, device, surface — is prepended to the sequence. Context tokens must be added to the vocabulary with the same `find_new_tokens` approach.

In [ ]:
# Listing 9.10: Generating a time-of-day context token
from datetime import datetime

def get_time_token(timestamp):
    hour = datetime.fromtimestamp(timestamp).hour
    if 6 <= hour < 12:
        return "TIME_MORNING"
    elif 12 <= hour < 18:
        return "TIME_AFTERNOON"
    elif 18 <= hour < 23:
        return "TIME_EVENING"
    else:
        return "TIME_NIGHT"

import time
print(get_time_token(time.time()))

### 9.2.3 Adding User Context

User attributes are prepended the same way. **Token dropout** (10% by default) randomly omits context tokens during training, so the model never becomes dependent on a feature that might be missing at inference time. Genuinely unknown values get an explicit `_UNK` marker so the sequence structure stays consistent.

In [ ]:
# Listing 9.11: Example user profile data
rich_data = [
    {"uid": 1, "age": 25, "loc": "US", "history": ["uuid_101", "uuid_102"]},
    {"uid": 2, "age": 10, "loc": "UK", "history": ["uuid_050", "uuid_051"]},
    {"uid": 3, "history": ["uuid_105", "uuid_101"]}  #A
]
#A No age or location available

In [ ]:
# Listing 9.12: Prepending user attributes to the sequence
class ContextualFormatter(BaseFormatter):  #A
    def __init__(self, item_df, dropout_rate=0.1):
        super().__init__(item_df)
        self.dropout_rate = dropout_rate

    def format(self, user_record, is_training=True):
        tokens = []

        val_age = user_record.get('age')  #B
        if val_age is not None and not pd.isna(val_age):
            if is_training and random.random() < self.dropout_rate:  #C
                pass
            else:
                tokens.append(f"AGE_{int(val_age)}")
        else:
            tokens.append("AGE_UNK")  #D

        val_loc = user_record.get('loc')  #E
        if val_loc is not None and not pd.isna(val_loc):
            if is_training and random.random() < self.dropout_rate:
                pass
            else:
                tokens.append(f"LOC_{val_loc}")
        else:
            tokens.append("LOC_UNK")

        for item_uuid in user_record['history']:  #F
            tokens.extend(self.get_item_tokens(item_uuid))
        return " ".join(tokens)

#A Inherits from BaseFormatter
#B Check for age data
#C Token dropout (10%)
#D Explicit unknown marker
#E Same pattern for location
#F Append semantic IDs for history items

# Listing 9.13: Example formatted sequences
ctx_formatter = ContextualFormatter(item_df)
for record in rich_data:
    print(f"User {record['uid']}:", repr(ctx_formatter.format(record, is_training=False)))

### 9.2.4 Adding Explicit Feedback Tokens

Likes and dislikes become `SENTIMENT_POS` / `SENTIMENT_NEG` prefixes on the item they refer to. Items with no explicit feedback get no sentiment token.

In [ ]:
# Listing 9.14: Adding explicit feedback to prompts in the Formatter
class FeedbackFormatter(BaseFormatter):  #A
    def __init__(self, item_df):
        super().__init__(item_df)

    def format(self, user_record, is_training=True):
        tokens = []
        feedback = user_record.get('feedback', {})  #B
        for item_uuid in user_record['history']:
            sentiment = feedback.get(item_uuid, 'VIEWED')  #C
            if sentiment == 'dislike':
                tokens.append("SENTIMENT_NEG")  #D
            elif sentiment == 'like':
                tokens.append("SENTIMENT_POS")
            tokens.extend(self.get_item_tokens(item_uuid))  #E
        return " ".join(tokens)

#A Inherits from BaseFormatter
#B Get feedback dictionary, default empty
#C Default to 'VIEWED' if no explicit signal
#D Prepend sentiment token
#E Semantic IDs follow immediately

fb_formatter = FeedbackFormatter(item_df)
demo = train_data.iloc[0].copy()
demo['feedback'] = {demo['history'][0]: 'like', demo['history'][1]: 'dislike'}
print(fb_formatter.format(demo)[:140], "...")

## 9.4 Generating Recommendations

Three steps: format the user's history into a prompt, ask the model to continue the sequence, parse the generated tokens into complete semantic IDs. `do_sample=True` with `top_k=50` and `temperature=0.8` balances diversity and quality; for deterministic debugging output, use beam search (`num_beams=5, do_sample=False`).

In [ ]:
# Listing 9.16: Generating semantic ID recommendations
def generate_recommendations(model, tokenizer, formatter,
                             user_data, num_items=3):
    model.eval()
    device = model.device
    prompt_text = formatter.format(user_data, is_training=False)  #A
    inputs = tokenizer(prompt_text, return_tensors="pt").to(device)

    max_new_tokens = num_items * 4  #B
    with torch.no_grad():
        output_ids = model.generate(
            inputs["input_ids"],
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True,  #C
            top_k=50,
            temperature=0.8
        )

    full_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    generated_part = full_text[len(prompt_text):].strip()  #D

    tokens = generated_part.split()
    items = []
    current = []
    for t in tokens:
        if t.startswith(("L1_", "L2_", "L3_", "LF_")):  #E
            current.append(t)
            if t.startswith("LF_"):  #F
                items.append(" ".join(current))
                current = []
                if len(items) >= num_items:
                    break
    return items

#A Format user history using the same formatter used during training
#B Each item is four tokens (L1, L2, L3, LF)
#C Enable sampling for diverse recommendations
#D Isolate newly generated tokens from the prompt
#E Only collect valid semantic ID tokens
#F Leaf node signals end of one item

In [ ]:
# Listing 9.17: Mapping generated IDs back to catalog items
class SemanticDecoder:
    def __init__(self, item_df):
        self.id_to_title = dict(
            zip(item_df['final_id'], item_df['title'])
        )  #A

    def parse_tokens(self, token_string):  #B
        try:
            ids = [int(t.split('_')[1]) for t in token_string.split()]
            return tuple(ids)
        except (IndexError, ValueError):
            return None

    def decode(self, token_string):
        semantic_tuple = self.parse_tokens(token_string)  #C
        if not semantic_tuple:
            return "Error: Could not parse tokens"
        if semantic_tuple in self.id_to_title:
            return self.id_to_title[semantic_tuple]  #D
        else:
            return f"Hallucination: {semantic_tuple} not in catalog"  #E

#A Pre-computed reverse index: tuple -> title
#B Parse "L1_6 L2_1 L3_81 LF_0" -> (6, 1, 81, 0)
#C Convert token string to tuple
#D Exact match found
#E Generated ID does not exist in catalog

In [ ]:
# Listing 9.18: Generating and decoding recommendations for a user
formatter = BaseFormatter(item_df)
decoder = SemanticDecoder(item_df)
id_to_title = dict(zip(item_df['id'], item_df['title']))  #A

user_inx = 6
user = train_data.iloc[user_inx]

print(f"User {user_inx} - Recent history:")
for i, raw_id in enumerate(user['history'][-4:]):  #B
    title = id_to_title.get(raw_id, "Unknown")
    print(f"  {i+1}. {title}")

recs = generate_recommendations(
    model, tokenizer, formatter, user, num_items=3
)  #C

print("\n--- Recommendations ---")
for i, raw_id in enumerate(recs):
    title = decoder.decode(raw_id)  #D
    print(f"  {i+1}. {title} ({raw_id})")

#A Separate lookup for displaying history (UUID -> title)
#B Show the user's last four watched items
#C Generate three recommendations
#D Decode each semantic ID to a title

## 9.5 Evaluating Generative Recommendations

Two complementary evaluators, both using leave-one-out with negative sampling:

- **GenerativeEvaluator** — the strictest end-to-end test: generate K items and check for an exact four-token match with the held-out item. Slow, use for final validation.
- **PerplexityEvaluator** — score 1 target + 99 negatives by model surprise. Deterministic and fast; the workhorse for model comparison during development.

In [ ]:
# Listing 9.19: GenerativeEvaluator - hit rate via leave-one-out
class GenerativeEvaluator:
    def __init__(self, model, tokenizer, formatter, k=10):
        self.model = model
        self.tokenizer = tokenizer
        self.formatter = formatter
        self.k = k

    def evaluate_user(self, user_data):
        full_history = user_data['history']
        if len(full_history) < 2:
            return 0, 0
        train_history = full_history[-11:-1]  #A
        target_uuid = full_history[-1]  #B
        target_tokens = self.formatter.get_item_tokens(target_uuid)
        if not target_tokens:
            return 0, 0
        target_string = " ".join(target_tokens)

        known_items = [t for t in train_history
                       if t in self.formatter.item_map]
        if len(known_items) == 0:
            return 0, 0
        if target_uuid not in self.formatter.item_map:
            return 0, 0

        eval_user = user_data.copy()
        eval_user['history'] = train_history

        recs = generate_recommendations(  #C
            self.model, self.tokenizer, self.formatter,
            eval_user, num_items=self.k
        )

        hit = 0
        ndcg = 0
        if target_string in recs:
            hit = 1
            rank = recs.index(target_string)  #D
            ndcg = 1.0 / np.log2(rank + 2)  #E
        return hit, ndcg

    def run_benchmark(self, test_data, limit=100):
        total_hr = []
        total_ndcg = []
        for i in range(min(limit, len(test_data))):
            hr, ndcg = self.evaluate_user(test_data.iloc[i])
            total_hr.append(hr)
            total_ndcg.append(ndcg)
        return {
            "k": self.k,
            "hit_rate": np.mean(total_hr),
            "NDCG": np.mean(total_ndcg),
        }

#A Last 10 items as context
#B The held-out item we want the model to predict
#C Generate K recommendations
#D Find where the target appears in the list
#E NDCG: position 0 -> 1.0, position 1 -> 0.63, etc.

gen_eval = GenerativeEvaluator(model, tokenizer, formatter, k=10)
print(gen_eval.run_benchmark(test_data, limit=50))

In [ ]:
# Listing 9.20: PerplexityEvaluator - scoring candidates by model surprise
class PerplexityEvaluator:
    def __init__(self, model, tokenizer, formatter,
                 all_items_set, k=10):
        self.model = model
        self.tokenizer = tokenizer
        self.formatter = formatter
        self.all_items = list(all_items_set)
        self.k = k
        self.device = model.device

    def calculate_score(self, history_str, candidate_str):
        full_text = (history_str + " " + candidate_str
                     + " <|endoftext|>")  #A
        inputs = self.tokenizer(
            full_text, return_tensors="pt"
        ).to(self.device)
        input_ids = inputs.input_ids

        history_ids = self.tokenizer.encode(
            history_str, add_special_tokens=False
        )
        start_idx = len(history_ids)  #B

        with torch.no_grad():
            outputs = self.model(input_ids, labels=input_ids)

        shift_logits = outputs.logits[..., :-1, :].contiguous()  #C
        shift_labels = input_ids[..., 1:].contiguous()

        target_logits = shift_logits[:, start_idx:, :]  #D
        target_labels = shift_labels[:, start_idx:]

        loss = F.cross_entropy(
            target_logits.transpose(1, 2),
            target_labels,
            reduction='sum'  #E
        )
        return -loss.item()  #F

    def evaluate_user(self, user_data):
        full_history = user_data['history']
        if len(full_history) < 2:
            return 0, 0
        train_history = full_history[:-1]  #G
        target_item = full_history[-1]

        eval_user = user_data.copy()
        eval_user['history'] = train_history
        history_str = self.formatter.format(
            eval_user, is_training=False
        )

        negatives = []
        hist_set = set(full_history)
        while len(negatives) < 99:  #H
            item = np.random.choice(self.all_items)
            if item not in hist_set:
                negatives.append(item)

        candidates = [target_item] + negatives  #I
        scores = []
        for item_uuid in candidates:
            item_tokens = self.formatter.get_item_tokens(item_uuid)
            if not item_tokens:
                scores.append(-9999)
                continue
            item_str = " ".join(item_tokens)
            score = self.calculate_score(history_str, item_str)
            scores.append(score)

        scores = np.array(scores)
        sorted_indices = np.argsort(scores)[::-1]  #J
        hit_rate = 1 if 0 in sorted_indices[:self.k] else 0  #K
        ndcg = 0
        pos = np.where(sorted_indices == 0)[0]
        if len(pos) > 0 and pos[0] < self.k:
            ndcg = 1.0 / np.log2(pos[0] + 2)  #L
        return hit_rate, ndcg

    def run_benchmark(self, test_data, limit=100):
        total_hr = []
        total_ndcg = []
        for i in range(min(limit, len(test_data))):
            hr, ndcg = self.evaluate_user(test_data.iloc[i])
            total_hr.append(hr)
            total_ndcg.append(ndcg)
        return {
            "k": self.k,
            "hit_rate": np.mean(total_hr),
            "NDCG": np.mean(total_ndcg),
        }

#A Concatenate history, candidate, and end-of-text marker
#B Find where the candidate tokens start
#C Shift logits and labels for next-token prediction
#D Slice to candidate tokens only
#E Sum, not mean: mean would bias toward shorter sequences
#F Negative loss: lower loss = higher score
#G Everything except the last item is context
#H Sample 99 negative items
#I True target is always at index 0
#J Sort by score, highest first
#K Check if target (index 0) is in top K
#L NDCG based on position

all_items = set(item_df['id'])
ppl_eval = PerplexityEvaluator(model, tokenizer, formatter, all_items, k=10)
print(ppl_eval.run_benchmark(test_data, limit=25))

## Where to next

- **`gen_recsys_alignment.ipynb`** — contrastive loss, DPO alignment, ICL reranking, anchored prompts, and sampled softmax (sections 9.3, 9.6–9.9).
- The next chapter builds a RAG-based recommender where the LLM reasons in natural language and a retrieval layer handles catalog grounding — dissolving the semantic gap instead of bridging it.